[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/onnx/tutorials/blob/main/02_Introduction_to_ONNX/03_ONNX_Ecosystem_Overview/ONNX_Ecosystem_Overview_Apply.ipynb)

# 1.3 ONNX Ecosystem Overview — Hands-On Practice

This notebook provides **hands-on experience** with the core components of the ONNX ecosystem. You will build models, convert them from multiple frameworks, run inference with ONNX Runtime, apply graph simplification, inspect model internals programmatically, and benchmark performance — all using real, runnable code.

## Table of Contents
1. [Setup and Installation](#section-1)
2. [Building an ONNX Model from Scratch](#section-2)
3. [Converting scikit-learn Models with skl2onnx](#section-3)
4. [Converting PyTorch Models with torch.onnx](#section-4)
5. [Running Inference with ONNX Runtime](#section-5)
6. [Programmatic Graph Inspection](#section-6)
7. [Visualizing Model Structure with Matplotlib](#section-7)
8. [Applying onnx-simplifier](#section-8)
9. [Before/After Simplification Comparison](#section-9)
10. [Performance Benchmarking](#section-10)
11. [Building a NetworkX Visualization of an ONNX Graph](#section-11)
12. [End-to-End Pipeline: Train → Export → Optimize → Deploy](#section-12)
13. [Summary](#section-13)

In [ ]:
# Install all ecosystem packages we'll use
!pip install onnx onnxruntime numpy matplotlib networkx scikit-learn skl2onnx onnxconverter-common -q

# Optional: install onnx-simplifier and onnxoptimizer (may fail in some environments)
!pip install onnx-simplifier onnxoptimizer -q 2>/dev/null || echo "Optional packages not available"

import warnings
warnings.filterwarnings('ignore')

<a id='section-2'></a>
## Section 2: Building an ONNX Model from Scratch

Before using converters, let's understand the ONNX model structure by building one manually using the `onnx` Python API. We'll construct a **two-layer MLP** (multi-layer perceptron) for binary classification:

$$\hat{y} = \sigma(W_2 \cdot \text{ReLU}(W_1 \cdot x + b_1) + b_2)$$

where $\sigma$ is the sigmoid function, $W_1 \in \mathbb{R}^{d \times h}$, $W_2 \in \mathbb{R}^{h \times 1}$, $x \in \mathbb{R}^{d}$.

Building a model manually forces you to think about every detail: tensor names, shapes, data types, operator attributes, and the order of nodes in the topological sort. This is the same structure that converters generate automatically — but understanding it deeply helps you debug conversion failures.

### The ONNX Construction API

The `onnx.helper` module provides factory functions for creating protobuf objects:

| Function | Creates | Key Arguments |
|:---|:---|:---|
| `make_tensor_value_info()` | ValueInfoProto | name, type, shape |
| `make_node()` | NodeProto | op_type, inputs, outputs, attributes |
| `make_graph()` | GraphProto | nodes, name, inputs, outputs, initializers |
| `make_model()` | ModelProto | graph, opset_imports |
| `numpy_helper.from_array()` | TensorProto | numpy array, name |

In [ ]:
import numpy as np
import onnx
from onnx import helper, TensorProto, numpy_helper
from onnx.checker import check_model

print(f'ONNX version: {onnx.__version__}')

# Model dimensions
input_dim = 4
hidden_dim = 8
output_dim = 1
batch = 'batch'  # symbolic dimension for dynamic batching

# Step 1: Create weight initializers (learned parameters)
np.random.seed(42)
W1_data = np.random.randn(input_dim, hidden_dim).astype(np.float32) * 0.5
b1_data = np.zeros(hidden_dim, dtype=np.float32)
W2_data = np.random.randn(hidden_dim, output_dim).astype(np.float32) * 0.5
b2_data = np.zeros(output_dim, dtype=np.float32)

W1 = numpy_helper.from_array(W1_data, name='W1')
b1 = numpy_helper.from_array(b1_data, name='b1')
W2 = numpy_helper.from_array(W2_data, name='W2')
b2 = numpy_helper.from_array(b2_data, name='b2')

# Step 2: Define operator nodes (topological order)
matmul1 = helper.make_node('MatMul', ['X', 'W1'], ['H1_pre'], name='layer1_matmul')
add1 = helper.make_node('Add', ['H1_pre', 'b1'], ['H1_bias'], name='layer1_add')
relu = helper.make_node('Relu', ['H1_bias'], ['H1'], name='layer1_relu')
matmul2 = helper.make_node('MatMul', ['H1', 'W2'], ['H2_pre'], name='layer2_matmul')
add2 = helper.make_node('Add', ['H2_pre', 'b2'], ['logit'], name='layer2_add')
sigmoid = helper.make_node('Sigmoid', ['logit'], ['Y'], name='output_sigmoid')

# Step 3: Define input/output tensor specs
X_info = helper.make_tensor_value_info('X', TensorProto.FLOAT, [batch, input_dim])
Y_info = helper.make_tensor_value_info('Y', TensorProto.FLOAT, [batch, output_dim])

# Step 4: Assemble graph
graph = helper.make_graph(
    nodes=[matmul1, add1, relu, matmul2, add2, sigmoid],
    name='binary_classifier',
    inputs=[X_info],
    outputs=[Y_info],
    initializer=[W1, b1, W2, b2]
)

# Step 5: Create model with opset 17
model = helper.make_model(graph, opset_imports=[helper.make_opsetid('', 17)])
model.ir_version = 8
model.producer_name = 'onnx-tutorial'
model.doc_string = 'Two-layer MLP for binary classification'

# Step 6: Validate
check_model(model)
print('Model validated successfully!')

# Save for later use
onnx.save(model, 'manual_mlp.onnx')

# Display model structure
print(f'\nModel: {model.graph.name}')
print(f'IR Version: {model.ir_version}, OpSet: {model.opset_import[0].version}')
print(f'Inputs: {[i.name for i in model.graph.input]}')
print(f'Outputs: {[o.name for o in model.graph.output]}')
print(f'Initializers: {[w.name for w in model.graph.initializer]}')
print(f'\nComputation graph ({len(model.graph.node)} nodes):')
for i, node in enumerate(model.graph.node):
    print(f'  [{i}] {node.name}: {node.op_type}({", ".join(node.input)}) -> {list(node.output)}')

# Verify with numpy
x_test = np.random.randn(3, input_dim).astype(np.float32)
h1 = np.maximum(0, x_test @ W1_data + b1_data)   # ReLU(X @ W1 + b1)
logit = h1 @ W2_data + b2_data                     # H1 @ W2 + b2
y_numpy = 1 / (1 + np.exp(-logit))                 # sigmoid

print(f'\nNumPy verification (batch of 3):')
print(f'  Input shape:  {x_test.shape}')
print(f'  Output shape: {y_numpy.shape}')
print(f'  Predictions:  {y_numpy.flatten().round(4)}')

<a id='section-3'></a>
## Section 3: Converting scikit-learn Models with skl2onnx

The `skl2onnx` converter handles scikit-learn's unique challenge: sklearn models are **not neural networks**. They include decision trees, SVMs, ensemble methods, and preprocessing pipelines — all requiring specialized ONNX operators from the `ai.onnx.ml` domain.

The conversion process maps each sklearn estimator to a specific ONNX ML operator:

| sklearn Estimator | ONNX ML Operator | Notes |
|:---|:---|:---|
| `LinearRegression` | `LinearRegressor` | Direct weight mapping |
| `LogisticRegression` | `LinearClassifier` | Includes probability calibration |
| `RandomForestClassifier` | `TreeEnsembleClassifier` | All trees serialized |
| `GradientBoostingRegressor` | `TreeEnsembleRegressor` | Boosted tree ensemble |
| `StandardScaler` | `Scaler` | Mean/variance normalization |
| `Pipeline` | Chained operators | Each step converted separately |

### The Conversion API

The key function is `convert_sklearn(model, initial_types)` where `initial_types` specifies the input tensor schema. This is necessary because sklearn models don't carry input shape information — they infer it at fit time but don't store it in a format that ONNX can use directly.

### Numerical Equivalence

After conversion, we verify numerical equivalence:

$$\|f_{\text{sklearn}}(\mathbf{x}) - f_{\text{onnx}}(\mathbf{x})\|_\infty \leq \epsilon$$

For sklearn models, $\epsilon$ should be exactly $0$ for deterministic models (linear, tree-based) and near-zero for models involving floating-point aggregation (ensembles with many trees).

In [ ]:
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import numpy as np
import onnx

# Create a classification dataset
X_data, y_data = make_classification(
    n_samples=500, n_features=4, n_classes=2, random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(
    X_data, y_data, test_size=0.2, random_state=42
)

# Build an sklearn pipeline
sklearn_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(n_estimators=10, max_depth=5, random_state=42))
])
sklearn_pipeline.fit(X_train, y_train)

sklearn_acc = sklearn_pipeline.score(X_test, y_test)
print(f'sklearn accuracy: {sklearn_acc:.4f}')

# Convert to ONNX using skl2onnx
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

initial_type = [('X', FloatTensorType([None, 4]))]
onnx_model = convert_sklearn(sklearn_pipeline, initial_types=initial_type,
                              target_opset=17)

# Validate and save
onnx.checker.check_model(onnx_model)
onnx.save(onnx_model, 'sklearn_rf_pipeline.onnx')

print(f'\nONNX model created successfully!')
print(f'  Graph name: {onnx_model.graph.name}')
print(f'  Nodes: {len(onnx_model.graph.node)}')
print(f'  Initializers: {len(onnx_model.graph.initializer)}')
print(f'  OpSet imports:')
for opset in onnx_model.opset_import:
    domain = opset.domain if opset.domain else 'default (ai.onnx)'
    print(f'    {domain}: version {opset.version}')

# List the operator types used
op_types = set(node.op_type for node in onnx_model.graph.node)
print(f'\n  Operator types used: {sorted(op_types)}')

In [ ]:
import onnxruntime as ort
import numpy as np

# Run ONNX model
sess = ort.InferenceSession('sklearn_rf_pipeline.onnx',
                            providers=['CPUExecutionProvider'])

# Compare predictions
X_test_f32 = X_test.astype(np.float32)
sklearn_preds = sklearn_pipeline.predict(X_test)
sklearn_proba = sklearn_pipeline.predict_proba(X_test)

onnx_results = sess.run(None, {'X': X_test_f32})
onnx_preds = onnx_results[0]
onnx_proba = onnx_results[1]

# Numerical comparison
pred_match = (sklearn_preds == onnx_preds.flatten()).mean()
print(f'Prediction agreement: {pred_match:.2%}')

# Compare probabilities
if isinstance(onnx_proba, list):
    onnx_proba_arr = np.array([[d[0], d[1]] for d in onnx_proba])
else:
    onnx_proba_arr = onnx_proba
max_diff = np.max(np.abs(sklearn_proba - onnx_proba_arr))
print(f'Max probability difference: {max_diff:.2e}')
print(f'Numerical equivalence (eps < 1e-5): {max_diff < 1e-5}')

print(f'\nSample predictions (first 5):')
print(f'  sklearn: {sklearn_preds[:5]}')
print(f'  ONNX:    {onnx_preds.flatten()[:5].astype(int)}')
print(f'\n  sklearn proba[0]: {sklearn_proba[:3, 1].round(4)}')
print(f'  ONNX proba[0]:    {onnx_proba_arr[:3, 1].round(4)}')

<a id='section-4'></a>
## Section 4: Converting PyTorch Models with torch.onnx

PyTorch's ONNX exporter uses **tracing** to capture the computation graph. The exporter runs the model with sample inputs, recording every tensor operation as an ONNX node. This approach is straightforward but has important implications:

### Tracing Semantics

When `torch.onnx.export` traces a model:

1. The model is executed with the provided `dummy_input` in evaluation mode
2. Every PyTorch operation (`torch.matmul`, `F.relu`, etc.) is mapped to ONNX operators
3. The resulting ONNX graph is **static** — it captures the exact execution path taken with the dummy input
4. Data-dependent control flow (e.g., `if x.sum() > 0`) is resolved for the dummy input only

### Dynamic Axes

By default, traced models have fixed input/output shapes. To support variable batch sizes or sequence lengths, you must specify `dynamic_axes`:

```python
dynamic_axes = {
    'input': {0: 'batch_size'},       # first dim is dynamic
    'output': {0: 'batch_size'}       # first dim is dynamic
}
```

This replaces fixed dimension values with symbolic names in the ONNX model, allowing the runtime to handle inputs of any size along those axes.

### Common Export Pitfalls

| Issue | Cause | Solution |
|:---|:---|:---|
| Wrong output values | Model not in eval mode | Call `model.eval()` before export |
| Missing operations | In-place ops not traced | Use out-of-place alternatives |
| Fixed batch size | Dynamic axes not specified | Set `dynamic_axes` parameter |
| Unsupported op | Custom autograd function | Register custom op or decompose |
| Shape mismatch | Hardcoded tensor sizes | Use relative sizing |

We'll demonstrate with a convolutional neural network (CNN) for image classification.

In [ ]:
try:
    import torch
    import torch.nn as nn
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False
    print("PyTorch not installed. Demonstrating with a manually-built CNN in ONNX.")

if HAS_TORCH:
    class SimpleCNN(nn.Module):
        def __init__(self, num_classes=10):
            super().__init__()
            self.features = nn.Sequential(
                nn.Conv2d(1, 16, 3, padding=1),
                nn.BatchNorm2d(16),
                nn.ReLU(),
                nn.MaxPool2d(2),
                nn.Conv2d(16, 32, 3, padding=1),
                nn.BatchNorm2d(32),
                nn.ReLU(),
                nn.AdaptiveAvgPool2d(1),
            )
            self.classifier = nn.Sequential(
                nn.Flatten(),
                nn.Linear(32, num_classes),
            )

        def forward(self, x):
            x = self.features(x)
            x = self.classifier(x)
            return x

    model = SimpleCNN(num_classes=10)
    model.eval()

    dummy_input = torch.randn(1, 1, 28, 28)

    torch.onnx.export(
        model, dummy_input, 'pytorch_cnn.onnx',
        opset_version=17,
        input_names=['image'],
        output_names=['logits'],
        dynamic_axes={
            'image': {0: 'batch_size'},
            'logits': {0: 'batch_size'}
        },
        do_constant_folding=True,
    )

    # Verify
    import onnx
    onnx_model = onnx.load('pytorch_cnn.onnx')
    onnx.checker.check_model(onnx_model)

    print('PyTorch CNN exported to ONNX successfully!')
    print(f'  Nodes: {len(onnx_model.graph.node)}')
    print(f'  Initializers (weights): {len(onnx_model.graph.initializer)}')
    print(f'\n  Operator types:')
    from collections import Counter
    op_counts = Counter(n.op_type for n in onnx_model.graph.node)
    for op, count in op_counts.most_common():
        print(f'    {op}: {count}')

    # Numerical verification
    with torch.no_grad():
        torch_out = model(dummy_input).numpy()

    sess = ort.InferenceSession('pytorch_cnn.onnx', providers=['CPUExecutionProvider'])
    onnx_out = sess.run(None, {'image': dummy_input.numpy()})[0]

    max_diff = np.max(np.abs(torch_out - onnx_out))
    print(f'\n  Max difference (PyTorch vs ONNX): {max_diff:.2e}')
    print(f'  Numerically equivalent: {max_diff < 1e-5}')

else:
    import onnx
    from onnx import helper, TensorProto, numpy_helper

    np.random.seed(42)
    conv_w = numpy_helper.from_array(np.random.randn(16, 1, 3, 3).astype(np.float32) * 0.1, 'conv1_w')
    conv_b = numpy_helper.from_array(np.zeros(16, dtype=np.float32), 'conv1_b')

    nodes = [
        helper.make_node('Conv', ['image', 'conv1_w', 'conv1_b'], ['conv1_out'],
                         kernel_shape=[3, 3], pads=[1, 1, 1, 1]),
        helper.make_node('Relu', ['conv1_out'], ['relu1_out']),
        helper.make_node('GlobalAveragePool', ['relu1_out'], ['gap_out']),
        helper.make_node('Flatten', ['gap_out'], ['flat_out']),
    ]

    X = helper.make_tensor_value_info('image', TensorProto.FLOAT, ['batch', 1, 28, 28])
    Y = helper.make_tensor_value_info('flat_out', TensorProto.FLOAT, ['batch', 16])
    graph = helper.make_graph(nodes, 'simple_cnn', [X], [Y], [conv_w, conv_b])
    model = helper.make_model(graph, opset_imports=[helper.make_opsetid('', 17)])
    onnx.checker.check_model(model)
    onnx.save(model, 'pytorch_cnn.onnx')

    print('Built CNN model manually (PyTorch not available)')
    print(f'  Nodes: {len(model.graph.node)}')
    for n in model.graph.node:
        print(f'    {n.op_type}: {list(n.input)} -> {list(n.output)}')

<a id='section-5'></a>
## Section 5: Running Inference with ONNX Runtime

ONNX Runtime (ORT) is the primary inference engine for ONNX models. The inference workflow follows three steps:

```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  Load Model     │────▶│  Create Session  │────▶│  Run Inference  │
│  (.onnx file)   │     │  (with EPs)      │     │  (feed inputs)  │
└─────────────────┘     └──────────────────┘     └─────────────────┘
```

### Session Options

ORT sessions are configured with `SessionOptions` that control:

- **Graph optimization level**: `ORT_DISABLE_ALL`, `ORT_ENABLE_BASIC`, `ORT_ENABLE_EXTENDED`, `ORT_ENABLE_ALL`
- **Execution mode**: Sequential vs Parallel
- **Inter/Intra-op thread count**: Controls parallelism
- **Memory optimization**: Enable/disable memory patterns, arena allocation
- **Profiling**: Enable to get per-node timing

### Execution Providers

The `providers` parameter specifies which hardware backends to use, in priority order:

```python
# Priority: try TensorRT first, fall back to CUDA, then CPU
providers = ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']
```

Each provider handles a subset of operators; unhandled operators fall through to the next provider in the list.

In [ ]:
import onnxruntime as ort
import numpy as np
import time

print(f'ONNX Runtime version: {ort.__version__}')
print(f'Available providers: {ort.get_available_providers()}')

# Session options for tuning
so = ort.SessionOptions()
so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
so.intra_op_num_threads = 4
so.inter_op_num_threads = 2
so.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL

# Load and run the manually-built MLP
sess_mlp = ort.InferenceSession('manual_mlp.onnx', sess_options=so,
                                 providers=['CPUExecutionProvider'])

print('\n--- Manual MLP Model ---')
print(f'Input:  {sess_mlp.get_inputs()[0].name}, shape={sess_mlp.get_inputs()[0].shape}, type={sess_mlp.get_inputs()[0].type}')
print(f'Output: {sess_mlp.get_outputs()[0].name}, shape={sess_mlp.get_outputs()[0].shape}, type={sess_mlp.get_outputs()[0].type}')

# Run inference with different batch sizes (dynamic batch!)
for batch_size in [1, 5, 10]:
    x = np.random.randn(batch_size, 4).astype(np.float32)
    result = sess_mlp.run(None, {'X': x})
    print(f'\n  Batch {batch_size}: input {x.shape} -> output {result[0].shape}')
    print(f'  Predictions: {result[0].flatten()[:5].round(4)}')

# Load and run the sklearn model
sess_rf = ort.InferenceSession('sklearn_rf_pipeline.onnx',
                                providers=['CPUExecutionProvider'])

print('\n--- sklearn Random Forest Pipeline ---')
for inp in sess_rf.get_inputs():
    print(f'Input:  {inp.name}, shape={inp.shape}, type={inp.type}')
for out in sess_rf.get_outputs():
    print(f'Output: {out.name}, shape={out.shape}, type={out.type}')

x_sample = np.random.randn(3, 4).astype(np.float32)
labels, proba = sess_rf.run(None, {'X': x_sample})
print(f'\n  Predictions: {labels.flatten()}')

# Load CNN model
sess_cnn = ort.InferenceSession('pytorch_cnn.onnx',
                                 providers=['CPUExecutionProvider'])
print('\n--- CNN Model ---')
inp = sess_cnn.get_inputs()[0]
print(f'Input: {inp.name}, shape={inp.shape}, type={inp.type}')
out = sess_cnn.get_outputs()[0]
print(f'Output: {out.name}, shape={out.shape}, type={out.type}')

<a id='section-6'></a>
## Section 6: Programmatic Graph Inspection

The `onnx` Python API provides full access to the model's protobuf structure. This is essential for:

- **Debugging exports**: Check that operators, shapes, and connections are correct
- **Auditing models**: Verify model complexity, parameter count, and operator usage
- **Preprocessing for deployment**: Extract metadata for serving infrastructure
- **Custom transformations**: Write scripts that modify the graph programmatically

### The Inspection Hierarchy

```
ModelProto
├── ir_version, producer_name, doc_string
├── opset_import[] — which operator sets are used
└── GraphProto (graph)
    ├── name
    ├── input[] — ValueInfoProto (name, type, shape)
    ├── output[] — ValueInfoProto
    ├── initializer[] — TensorProto (weights)
    ├── node[] — NodeProto
    │   ├── op_type, name, domain
    │   ├── input[], output[]
    │   └── attribute[] — AttributeProto
    └── value_info[] — intermediate tensor info (after shape inference)
```

Each level of the hierarchy is accessible as Python attributes on the protobuf objects. The ONNX library also provides helper functions for common operations like converting between numpy arrays and TensorProto objects.

In [ ]:
import onnx
from onnx import numpy_helper, shape_inference
import numpy as np

# Load a model for inspection
model = onnx.load('manual_mlp.onnx')

print('=' * 60)
print('MODEL INSPECTION REPORT')
print('=' * 60)

# Model-level metadata
print(f'\n1. MODEL METADATA')
print(f'   IR Version: {model.ir_version}')
print(f'   Producer: {model.producer_name}')
print(f'   Doc: {model.doc_string}')
print(f'   OpSet: {[(op.domain or "default", op.version) for op in model.opset_import]}')

# Graph-level info
g = model.graph
print(f'\n2. GRAPH: "{g.name}"')
print(f'   Nodes: {len(g.node)}')
print(f'   Inputs: {len(g.input)}')
print(f'   Outputs: {len(g.output)}')
print(f'   Initializers: {len(g.initializer)}')

# Inputs
print(f'\n3. INPUTS')
for inp in g.input:
    t = inp.type.tensor_type
    shape = []
    for d in t.shape.dim:
        shape.append(d.dim_param if d.dim_param else d.dim_value)
    elem_type = onnx.TensorProto.DataType.Name(t.elem_type)
    print(f'   {inp.name}: {elem_type} {shape}')

# Outputs
print(f'\n4. OUTPUTS')
for out in g.output:
    t = out.type.tensor_type
    shape = []
    for d in t.shape.dim:
        shape.append(d.dim_param if d.dim_param else d.dim_value)
    elem_type = onnx.TensorProto.DataType.Name(t.elem_type)
    print(f'   {out.name}: {elem_type} {shape}')

# Initializers (weights)
print(f'\n5. INITIALIZERS (weights)')
total_params = 0
for init in g.initializer:
    arr = numpy_helper.to_array(init)
    total_params += arr.size
    print(f'   {init.name}: shape={list(arr.shape)}, dtype={arr.dtype}, '
          f'params={arr.size}, range=[{arr.min():.4f}, {arr.max():.4f}]')
print(f'   Total parameters: {total_params:,}')
print(f'   Model size estimate: {total_params * 4 / 1024:.1f} KB (FP32)')

# Nodes (operators)
print(f'\n6. COMPUTATION GRAPH')
for i, node in enumerate(g.node):
    attrs = {a.name: a for a in node.attribute}
    attr_str = ', '.join(f'{a.name}={a.i or a.f or a.s}' for a in node.attribute)
    print(f'   [{i}] {node.op_type} "{node.name}"')
    print(f'       inputs:  {list(node.input)}')
    print(f'       outputs: {list(node.output)}')
    if attr_str:
        print(f'       attrs:   {attr_str}')

# Shape inference
print(f'\n7. SHAPE INFERENCE')
inferred = shape_inference.infer_shapes(model)
for vi in inferred.graph.value_info:
    t = vi.type.tensor_type
    shape = []
    for d in t.shape.dim:
        shape.append(d.dim_param if d.dim_param else d.dim_value)
    print(f'   {vi.name}: {shape}')

<a id='section-7'></a>
## Section 7: Visualizing Model Structure with Matplotlib

While Netron provides interactive visualization, we can also render ONNX graphs programmatically using **matplotlib** and **networkx**. This is useful for:

- Automated documentation generation
- Integration into CI/CD pipelines (generate graph images on export)
- Custom visualizations that highlight specific aspects (e.g., quantized vs FP32 nodes)
- Environments where Netron isn't available (headless servers, notebooks)

Our approach:
1. Parse the ONNX graph into a NetworkX directed graph
2. Assign visual properties based on node type (operator, initializer, input, output)
3. Layout the graph using a hierarchical algorithm
4. Render with matplotlib, including tensor shapes and operator names

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
import onnx
import numpy as np

model = onnx.load('manual_mlp.onnx')
g = model.graph

fig, ax = plt.subplots(figsize=(14, 10))
ax.set_xlim(0, 14)
ax.set_ylim(0, 10)
ax.axis('off')

input_names = {i.name for i in g.input}
output_names = {o.name for o in g.output}
init_names = {i.name for i in g.initializer}

y_pos = 9.0
node_positions = {}

for inp in g.input:
    if inp.name not in init_names:
        t = inp.type.tensor_type
        shape = [d.dim_param or d.dim_value for d in t.shape.dim]
        node_positions[inp.name] = (2, y_pos)
        rect = FancyBboxPatch((1, y_pos - 0.25), 2, 0.5, boxstyle='round,pad=0.08',
                              facecolor='#98FB98', edgecolor='black', linewidth=1.5)
        ax.add_patch(rect)
        ax.text(2, y_pos, f'{inp.name}\n{shape}', ha='center', va='center', fontsize=8, fontweight='bold')

x_positions = [2, 5, 8, 11]  # initializer x-positions
init_y = 9.0
for idx, init in enumerate(g.initializer):
    arr = onnx.numpy_helper.to_array(init)
    x = x_positions[idx % len(x_positions)] + 3
    node_positions[init.name] = (x, init_y)
    rect = FancyBboxPatch((x - 1, init_y - 0.25), 2, 0.5, boxstyle='round,pad=0.08',
                          facecolor='#DDA0DD', edgecolor='black', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x, init_y, f'{init.name}\n{list(arr.shape)}', ha='center', va='center', fontsize=7, fontweight='bold')

op_y_start = 7.5
op_spacing = 1.2
for i, node in enumerate(g.node):
    y = op_y_start - i * op_spacing
    x = 7
    node_positions[node.name] = (x, y)

    rect = FancyBboxPatch((x - 1.5, y - 0.3), 3, 0.6, boxstyle='round,pad=0.1',
                          facecolor='#87CEEB', edgecolor='black', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x, y, f'{node.op_type}\n({node.name})', ha='center', va='center',
            fontsize=8, fontweight='bold')

    for out_name in node.output:
        node_positions[out_name] = (x, y)

for i, node in enumerate(g.node):
    y = op_y_start - i * op_spacing
    x = 7
    for inp_name in node.input:
        if inp_name in node_positions:
            sx, sy = node_positions[inp_name]
            ax.annotate('', xy=(x - 0.5, y + 0.3), xytext=(sx, sy - 0.25 if sy > y else sy + 0.3),
                       arrowprops=dict(arrowstyle='->', color='#666', lw=1.2,
                                      connectionstyle='arc3,rad=0.15'))

for out in g.output:
    y = op_y_start - (len(g.node) - 1) * op_spacing - 1.0
    rect = FancyBboxPatch((6, y - 0.25), 2, 0.5, boxstyle='round,pad=0.08',
                          facecolor='#FFFF99', edgecolor='black', linewidth=1.5)
    ax.add_patch(rect)
    t = out.type.tensor_type
    shape = [d.dim_param or d.dim_value for d in t.shape.dim]
    ax.text(7, y, f'{out.name}\n{shape}', ha='center', va='center', fontsize=8, fontweight='bold')

legend_elements = [
    mpatches.Patch(color='#98FB98', label='Input tensor'),
    mpatches.Patch(color='#DDA0DD', label='Initializer (weights)'),
    mpatches.Patch(color='#87CEEB', label='Operator node'),
    mpatches.Patch(color='#FFFF99', label='Output tensor'),
]
ax.legend(handles=legend_elements, loc='upper left', fontsize=9)
ax.set_title(f'ONNX Model Structure: {g.name}\n'
             f'({len(g.node)} ops, {sum(onnx.numpy_helper.to_array(i).size for i in g.initializer):,} params)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

<a id='section-8'></a>
## Section 8: Applying onnx-simplifier

`onnx-simplifier` is a tool that simplifies ONNX models by:

1. **Constant folding**: Evaluating subgraphs with all-constant inputs and replacing them with the computed constants
2. **Redundant node elimination**: Removing identity operations, unused nodes, and duplicate computations
3. **Shape inference propagation**: Resolving dynamic shapes where possible

The simplifier uses ONNX Runtime as an **oracle**: it feeds constant inputs through the model, captures intermediate values, and replaces computed subgraphs with constants. This is more aggressive than `onnxoptimizer` because it actually *runs* the model.

### When to Simplify

Simplification is most effective on models exported from PyTorch and TensorFlow, which often contain:

- **Shape computation subgraphs**: Operators like `Shape`, `Gather`, `Unsqueeze` that compute tensor dimensions at runtime, even when those dimensions are known at export time
- **Redundant cast operations**: Type conversions between identical types
- **Identity operations**: Nodes that pass through values unchanged
- **Constant reshape patterns**: Reshape operations where the target shape is a constant

### Before vs After

For a typical ResNet-50 export from PyTorch:

| Metric | Before | After | Reduction |
|:---|:---:|:---:|:---:|
| Nodes | ~180 | ~120 | 33% |
| Initializers | ~55 | ~55 | 0% |
| File size | 98 MB | 97 MB | ~1% |
| Shape ops | ~30 | 0 | 100% |

Note: file size reduction is small because weights dominate. The benefit is in graph complexity reduction, which speeds up runtime optimization.

In [ ]:
import onnx
from onnx import helper, TensorProto, numpy_helper
import numpy as np
from collections import Counter

# Build a model with redundancies that the simplifier can optimize
np.random.seed(42)

W = numpy_helper.from_array(np.random.randn(4, 8).astype(np.float32), 'W')
b = numpy_helper.from_array(np.random.randn(8).astype(np.float32), 'b')

# Deliberately add redundant operations:
# 1. Identity node
# 2. Reshape to same shape
# 3. Add zero (no-op)
# 4. Multiply by one (no-op)
zero_const = helper.make_node('Constant', [], ['zero'],
    value=helper.make_tensor('zero_val', TensorProto.FLOAT, [8],
                              np.zeros(8, dtype=np.float32)))
one_const = helper.make_node('Constant', [], ['one'],
    value=helper.make_tensor('one_val', TensorProto.FLOAT, [8],
                              np.ones(8, dtype=np.float32)))
shape_const = helper.make_node('Constant', [], ['target_shape'],
    value=helper.make_tensor('shape_val', TensorProto.INT64, [2],
                              np.array([1, 4], dtype=np.int64)))

nodes = [
    shape_const,
    helper.make_node('Reshape', ['X', 'target_shape'], ['X_reshaped']),  # reshape to same
    zero_const, one_const,
    helper.make_node('Identity', ['X_reshaped'], ['X_id']),              # identity
    helper.make_node('MatMul', ['X_id', 'W'], ['H_pre']),
    helper.make_node('Add', ['H_pre', 'b'], ['H_bias']),
    helper.make_node('Add', ['H_bias', 'zero'], ['H_plus_zero']),       # add zero
    helper.make_node('Mul', ['H_plus_zero', 'one'], ['H_times_one']),   # multiply one
    helper.make_node('Relu', ['H_times_one'], ['Y']),
]

X_info = helper.make_tensor_value_info('X', TensorProto.FLOAT, [1, 4])
Y_info = helper.make_tensor_value_info('Y', TensorProto.FLOAT, [1, 8])

graph = helper.make_graph(nodes, 'redundant_model', [X_info], [Y_info], [W, b])
raw_model = helper.make_model(graph, opset_imports=[helper.make_opsetid('', 17)])
onnx.checker.check_model(raw_model)
onnx.save(raw_model, 'model_raw.onnx')

print('BEFORE simplification:')
print(f'  Nodes: {len(raw_model.graph.node)}')
op_counts_before = Counter(n.op_type for n in raw_model.graph.node)
for op, count in op_counts_before.most_common():
    print(f'    {op}: {count}')

# Apply simplifier
try:
    from onnxsim import simplify
    simplified_model, check = simplify(raw_model)
    onnx.save(simplified_model, 'model_simplified.onnx')

    print(f'\nAFTER simplification:')
    print(f'  Nodes: {len(simplified_model.graph.node)}')
    op_counts_after = Counter(n.op_type for n in simplified_model.graph.node)
    for op, count in op_counts_after.most_common():
        print(f'    {op}: {count}')

    print(f'\n  Nodes removed: {len(raw_model.graph.node) - len(simplified_model.graph.node)}')
    print(f'  Simplification valid: {check}')

except ImportError:
    print('\nonnx-simplifier not available. Applying manual optimizations...')
    try:
        import onnxoptimizer as opt
        passes = opt.get_available_passes()
        simplified_model = opt.optimize(raw_model)
        onnx.save(simplified_model, 'model_simplified.onnx')

        print(f'\nAFTER onnxoptimizer:')
        print(f'  Nodes: {len(simplified_model.graph.node)}')
        op_counts_after = Counter(n.op_type for n in simplified_model.graph.node)
        for op, count in op_counts_after.most_common():
            print(f'    {op}: {count}')
    except ImportError:
        print('  Neither onnx-simplifier nor onnxoptimizer available.')
        simplified_model = raw_model

<a id='section-9'></a>
## Section 9: Before/After Simplification Comparison

A critical step after any model transformation is **numerical verification**: confirming that the simplified model produces the same outputs as the original model for the same inputs. We verify:

$$\forall \mathbf{x} \in \mathcal{X}_{\text{test}}: \|f_{\text{raw}}(\mathbf{x}) - f_{\text{simplified}}(\mathbf{x})\|_\infty \leq \epsilon$$

where $\epsilon$ is typically on the order of $10^{-6}$ for FP32 models (accounting for floating-point non-associativity introduced by operator reordering).

Beyond numerical equivalence, we compare structural metrics: node count, operator distribution, and model file size. These metrics quantify the simplification's effectiveness and help identify whether further optimization is worthwhile.

In [ ]:
import matplotlib.pyplot as plt
import onnxruntime as ort
import numpy as np
import os

model_raw = onnx.load('model_raw.onnx')

simplified_path = 'model_simplified.onnx'
if os.path.exists(simplified_path):
    model_simp = onnx.load(simplified_path)
else:
    model_simp = model_raw

# Numerical verification
sess_raw = ort.InferenceSession('model_raw.onnx', providers=['CPUExecutionProvider'])
sess_simp = ort.InferenceSession(simplified_path if os.path.exists(simplified_path)
                                  else 'model_raw.onnx',
                                  providers=['CPUExecutionProvider'])

n_tests = 100
max_diffs = []
np.random.seed(0)
for _ in range(n_tests):
    x = np.random.randn(1, 4).astype(np.float32)
    raw_out = sess_raw.run(None, {'X': x})[0]
    simp_out = sess_simp.run(None, {'X': x})[0]
    max_diffs.append(np.max(np.abs(raw_out - simp_out)))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Chart 1: Node count comparison
labels_raw = Counter(n.op_type for n in model_raw.graph.node)
labels_simp = Counter(n.op_type for n in model_simp.graph.node)
all_ops = sorted(set(labels_raw.keys()) | set(labels_simp.keys()))

x = np.arange(len(all_ops))
w = 0.35
bars1 = axes[0].bar(x - w/2, [labels_raw.get(op, 0) for op in all_ops], w,
                     label='Before', color='#FF6B6B', edgecolor='black')
bars2 = axes[0].bar(x + w/2, [labels_simp.get(op, 0) for op in all_ops], w,
                     label='After', color='#4ECDC4', edgecolor='black')
axes[0].set_xticks(x)
axes[0].set_xticklabels(all_ops, rotation=45, ha='right', fontsize=8)
axes[0].set_ylabel('Count')
axes[0].set_title('Operator Distribution', fontsize=11, fontweight='bold')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Chart 2: Model metrics comparison
metrics = ['Nodes', 'Initializers', 'Size (KB)']
raw_vals = [len(model_raw.graph.node), len(model_raw.graph.initializer),
            os.path.getsize('model_raw.onnx') / 1024]
simp_vals = [len(model_simp.graph.node), len(model_simp.graph.initializer),
             os.path.getsize(simplified_path if os.path.exists(simplified_path)
                            else 'model_raw.onnx') / 1024]

x2 = np.arange(len(metrics))
axes[1].bar(x2 - w/2, raw_vals, w, label='Before', color='#FF6B6B', edgecolor='black')
axes[1].bar(x2 + w/2, simp_vals, w, label='After', color='#4ECDC4', edgecolor='black')
axes[1].set_xticks(x2)
axes[1].set_xticklabels(metrics, fontsize=9)
axes[1].set_title('Model Metrics', fontsize=11, fontweight='bold')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

# Chart 3: Numerical difference distribution
axes[2].hist(max_diffs, bins=30, color='#45B7D1', edgecolor='black', alpha=0.85)
axes[2].axvline(x=np.mean(max_diffs), color='red', linestyle='--', label=f'mean={np.mean(max_diffs):.2e}')
axes[2].set_xlabel('Max Absolute Difference')
axes[2].set_ylabel('Frequency')
axes[2].set_title('Numerical Equivalence', fontsize=11, fontweight='bold')
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.suptitle('Before vs After Simplification', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'\nNumerical verification ({n_tests} random inputs):')
print(f'  Max difference: {max(max_diffs):.2e}')
print(f'  Mean difference: {np.mean(max_diffs):.2e}')
print(f'  All within tolerance (1e-5): {all(d < 1e-5 for d in max_diffs)}')

<a id='section-10'></a>
## Section 10: Performance Benchmarking

Performance benchmarking is essential for making informed runtime and optimization decisions. We measure three key metrics:

### Latency

$$L = \frac{1}{N} \sum_{i=1}^{N} t_i \quad \text{(mean inference time over N runs)}$$

We use warm-up runs to exclude JIT compilation and memory allocation overhead, then measure over many iterations for statistical stability.

### Throughput

$$\Theta = \frac{N_{\text{samples}}}{T_{\text{total}}} \quad \text{(samples per second)}$$

### Speedup

$$S = \frac{L_{\text{baseline}}}{L_{\text{optimized}}}$$

A speedup of $S > 1$ means the optimized configuration is faster.

### Best Practices for Benchmarking

1. **Warm-up**: Run 10-50 iterations before measuring to fill caches and trigger JIT
2. **Statistical significance**: Report mean, std, min, median, and percentiles (p50, p95, p99)
3. **Fixed inputs**: Use the same input data across all configurations
4. **Controlled environment**: Minimize background CPU load, pin threads if possible
5. **Multiple batch sizes**: Latency characteristics change dramatically with batch size

In [ ]:
import onnxruntime as ort
import numpy as np
import time

def benchmark_model(model_path, input_name, input_shape, n_warmup=50, n_runs=500):
    '''Benchmark an ONNX model with detailed statistics.'''
    sess = ort.InferenceSession(model_path, providers=['CPUExecutionProvider'])

    x = np.random.randn(*input_shape).astype(np.float32)

    # Warm-up
    for _ in range(n_warmup):
        sess.run(None, {input_name: x})

    # Benchmark
    latencies = []
    for _ in range(n_runs):
        start = time.perf_counter()
        sess.run(None, {input_name: x})
        latencies.append((time.perf_counter() - start) * 1000)  # ms

    latencies = np.array(latencies)
    return {
        'mean': np.mean(latencies),
        'std': np.std(latencies),
        'min': np.min(latencies),
        'median': np.median(latencies),
        'p95': np.percentile(latencies, 95),
        'p99': np.percentile(latencies, 99),
        'throughput': 1000 / np.mean(latencies),
        'latencies': latencies,
    }

# Benchmark all three models
models = {
    'MLP (manual)': ('manual_mlp.onnx', 'X', (1, 4)),
    'RF Pipeline': ('sklearn_rf_pipeline.onnx', 'X', (1, 4)),
    'CNN': ('pytorch_cnn.onnx', sess_cnn.get_inputs()[0].name,
            tuple(d if isinstance(d, int) else 1 for d in sess_cnn.get_inputs()[0].shape)),
}

results = {}
for name, (path, input_name, shape) in models.items():
    print(f'Benchmarking {name}...')
    results[name] = benchmark_model(path, input_name, shape)

print('\n' + '=' * 70)
print(f'{"Model":<18} {"Mean (ms)":<12} {"Std":<10} {"P95 (ms)":<12} {"Throughput":<12}')
print('=' * 70)
for name, r in results.items():
    print(f'{name:<18} {r["mean"]:<12.3f} {r["std"]:<10.3f} {r["p95"]:<12.3f} {r["throughput"]:<12.0f} samples/s')

# Benchmark with different batch sizes for MLP
batch_sizes = [1, 4, 8, 16, 32, 64, 128]
batch_results = {}
for bs in batch_sizes:
    batch_results[bs] = benchmark_model('manual_mlp.onnx', 'X', (bs, 4), n_warmup=20, n_runs=200)

print(f'\nMLP Batch Size Scaling:')
print(f'{"Batch":<8} {"Latency (ms)":<15} {"Throughput (samples/s)":<25}')
for bs, r in batch_results.items():
    throughput = bs * 1000 / r['mean']
    print(f'{bs:<8} {r["mean"]:<15.3f} {throughput:<25.0f}')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Chart 1: Latency comparison across models
model_names = list(results.keys())
means = [results[n]['mean'] for n in model_names]
stds = [results[n]['std'] for n in model_names]
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']

ax1 = axes[0, 0]
bars = ax1.bar(model_names, means, yerr=stds, capsize=5,
               color=colors, edgecolor='black', linewidth=0.8)
ax1.set_ylabel('Latency (ms)', fontsize=10)
ax1.set_title('Inference Latency by Model', fontsize=12, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)
for bar, mean in zip(bars, means):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{mean:.3f}ms', ha='center', va='bottom', fontsize=9, fontweight='bold')

# Chart 2: Latency distributions
ax2 = axes[0, 1]
for name, color in zip(model_names, colors):
    ax2.hist(results[name]['latencies'], bins=40, alpha=0.6, label=name,
             color=color, edgecolor='black', linewidth=0.5)
ax2.set_xlabel('Latency (ms)', fontsize=10)
ax2.set_ylabel('Frequency', fontsize=10)
ax2.set_title('Latency Distribution', fontsize=12, fontweight='bold')
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3)

# Chart 3: Batch size scaling
ax3 = axes[1, 0]
bs_list = list(batch_results.keys())
lat_list = [batch_results[bs]['mean'] for bs in bs_list]
throughput_list = [bs * 1000 / batch_results[bs]['mean'] for bs in bs_list]

ax3.plot(bs_list, lat_list, 'o-', color='#FF6B6B', linewidth=2, markersize=6, label='Latency')
ax3.set_xlabel('Batch Size', fontsize=10)
ax3.set_ylabel('Latency (ms)', fontsize=10, color='#FF6B6B')
ax3.tick_params(axis='y', labelcolor='#FF6B6B')
ax3.set_title('Batch Size Scaling (MLP)', fontsize=12, fontweight='bold')
ax3.grid(alpha=0.3)

ax3b = ax3.twinx()
ax3b.plot(bs_list, throughput_list, 's-', color='#4ECDC4', linewidth=2, markersize=6, label='Throughput')
ax3b.set_ylabel('Throughput (samples/s)', fontsize=10, color='#4ECDC4')
ax3b.tick_params(axis='y', labelcolor='#4ECDC4')

lines1, labels1 = ax3.get_legend_handles_labels()
lines2, labels2 = ax3b.get_legend_handles_labels()
ax3.legend(lines1 + lines2, labels1 + labels2, loc='center right', fontsize=8)

# Chart 4: Percentile comparison
ax4 = axes[1, 1]
percentile_labels = ['Min', 'P50', 'Mean', 'P95', 'P99']
x = np.arange(len(percentile_labels))
w = 0.25

for i, (name, color) in enumerate(zip(model_names, colors)):
    r = results[name]
    vals = [r['min'], r['median'], r['mean'], r['p95'], r['p99']]
    ax4.bar(x + i * w, vals, w, label=name, color=color, edgecolor='black', linewidth=0.5)

ax4.set_xticks(x + w)
ax4.set_xticklabels(percentile_labels, fontsize=9)
ax4.set_ylabel('Latency (ms)', fontsize=10)
ax4.set_title('Latency Percentiles', fontsize=12, fontweight='bold')
ax4.legend(fontsize=8)
ax4.grid(axis='y', alpha=0.3)

plt.suptitle('ONNX Runtime Performance Benchmarks', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

<a id='section-11'></a>
## Section 11: Building a NetworkX Visualization of an ONNX Graph

We now build a general-purpose function that converts any ONNX model into a NetworkX directed graph and renders it. This function can be reused across the tutorial to visualize any model.

### The Conversion Algorithm

```
For each node in the ONNX graph:
    1. Create a graph node for the operator
    2. For each input tensor:
       - If it's an initializer → create an initializer node and edge
       - If it's a graph input → create an input node and edge
       - If it's produced by another op → create an edge from that op
    3. For each output tensor:
       - If it's a graph output → create an output node and edge
       - Otherwise → record as intermediate (used by downstream ops)
```

This algorithm produces a bipartite-like graph with operator nodes and tensor nodes, connected by directed edges representing data flow.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
import onnx
from onnx import numpy_helper
import numpy as np

def onnx_to_networkx(model):
    '''Convert an ONNX model to a NetworkX directed graph.'''
    G = nx.DiGraph()
    g = model.graph

    init_names = {i.name for i in g.initializer}
    graph_inputs = {i.name for i in g.input} - init_names
    graph_outputs = {o.name for o in g.output}

    tensor_producer = {}

    for inp_name in graph_inputs:
        G.add_node(inp_name, node_type='input', label=inp_name)

    for init in g.initializer:
        arr = numpy_helper.to_array(init)
        label = f'{init.name}\n{list(arr.shape)}'
        G.add_node(init.name, node_type='initializer', label=label,
                   shape=list(arr.shape))

    for i, node in enumerate(g.node):
        node_id = f'{node.op_type}_{i}'
        G.add_node(node_id, node_type='operator', label=node.op_type,
                   op_type=node.op_type)

        for inp_name in node.input:
            if inp_name in graph_inputs or inp_name in init_names:
                G.add_edge(inp_name, node_id, tensor_name=inp_name)
            elif inp_name in tensor_producer:
                G.add_edge(tensor_producer[inp_name], node_id, tensor_name=inp_name)

        for out_name in node.output:
            tensor_producer[out_name] = node_id

            if out_name in graph_outputs:
                G.add_node(out_name, node_type='output', label=out_name)
                G.add_edge(node_id, out_name, tensor_name=out_name)

    return G


def visualize_onnx_graph(model, title=None, figsize=(14, 10)):
    '''Visualize an ONNX model as a NetworkX graph.'''
    G = onnx_to_networkx(model)

    color_map = {
        'input': '#98FB98',
        'initializer': '#DDA0DD',
        'operator': '#87CEEB',
        'output': '#FFFF99',
    }
    size_map = {
        'input': 2000,
        'initializer': 1500,
        'operator': 2500,
        'output': 2000,
    }

    node_colors = [color_map.get(G.nodes[n].get('node_type', 'operator'), '#E8E8E8') for n in G.nodes()]
    node_sizes = [size_map.get(G.nodes[n].get('node_type', 'operator'), 1500) for n in G.nodes()]
    labels = {n: G.nodes[n].get('label', n) for n in G.nodes()}

    try:
        pos = nx.nx_agraph.graphviz_layout(G, prog='dot')
    except:
        pos = nx.spring_layout(G, k=2, iterations=50, seed=42)

    fig, ax = plt.subplots(figsize=figsize)

    nx.draw(G, pos, ax=ax, with_labels=True, labels=labels,
            node_color=node_colors, node_size=node_sizes,
            font_size=7, font_weight='bold', arrows=True, arrowsize=15,
            edge_color='#999', connectionstyle='arc3,rad=0.1')

    edge_labels = {(u, v): d.get('tensor_name', '') for u, v, d in G.edges(data=True)}
    nx.draw_networkx_edge_labels(G, pos, edge_labels, font_size=5, alpha=0.6, ax=ax)

    legend_elements = [
        mpatches.Patch(color='#98FB98', label='Graph Input'),
        mpatches.Patch(color='#DDA0DD', label='Initializer (weights)'),
        mpatches.Patch(color='#87CEEB', label='Operator'),
        mpatches.Patch(color='#FFFF99', label='Graph Output'),
    ]
    ax.legend(handles=legend_elements, loc='upper left', fontsize=9)

    if title is None:
        title = f'ONNX Graph: {model.graph.name}'
    ax.set_title(title, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

    return G


# Visualize our manually-built MLP
model_mlp = onnx.load('manual_mlp.onnx')
G = visualize_onnx_graph(model_mlp, title='NetworkX Visualization: Binary Classifier MLP')

print(f'\nGraph Statistics:')
print(f'  Nodes: {G.number_of_nodes()}')
print(f'  Edges: {G.number_of_edges()}')
print(f'  Node types: {dict(Counter(G.nodes[n]["node_type"] for n in G.nodes()))}')
print(f'  Is DAG: {nx.is_directed_acyclic_graph(G)}')

if nx.is_directed_acyclic_graph(G):
    topo_order = list(nx.topological_sort(G))
    print(f'  Topological order: {" -> ".join(G.nodes[n]["label"] for n in topo_order[:8])}...')

<a id='section-12'></a>
## Section 12: End-to-End Pipeline: Train → Export → Optimize → Deploy

Let's put everything together in a single end-to-end pipeline that demonstrates the complete ONNX workflow:

```
┌─────────┐    ┌──────────┐    ┌──────────┐    ┌───────────┐    ┌──────────┐
│  Train  │───▶│  Convert  │───▶│ Validate │───▶│ Optimize  │───▶│  Deploy  │
│ (sklearn)│   │ (skl2onnx)│   │ (checker) │   │(simplify) │   │  (ORT)   │
└─────────┘    └──────────┘    └──────────┘    └───────────┘    └──────────┘
```

This pipeline mirrors production deployment workflows and exercises every major ecosystem component we've studied.

In [ ]:
import numpy as np
import time
from sklearn.datasets import make_classification
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import onnx
from onnx import shape_inference
import onnxruntime as ort

print('=' * 60)
print('END-TO-END ONNX DEPLOYMENT PIPELINE')
print('=' * 60)

# STAGE 1: Train
print('\n--- STAGE 1: Train ---')
X, y = make_classification(n_samples=1000, n_features=10, n_classes=3,
                           n_informative=6, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

clf = GradientBoostingClassifier(n_estimators=50, max_depth=4, random_state=42)
clf.fit(X_train, y_train)
train_acc = accuracy_score(y_test, clf.predict(X_test))
print(f'  sklearn accuracy: {train_acc:.4f}')

# STAGE 2: Convert
print('\n--- STAGE 2: Convert to ONNX ---')
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

initial_type = [('X', FloatTensorType([None, 10]))]
onnx_model = convert_sklearn(clf, initial_types=initial_type, target_opset=17)
print(f'  Nodes: {len(onnx_model.graph.node)}')
print(f'  Initializers: {len(onnx_model.graph.initializer)}')

# STAGE 3: Validate
print('\n--- STAGE 3: Validate ---')
onnx.checker.check_model(onnx_model)
print('  check_model: PASSED')

inferred = shape_inference.infer_shapes(onnx_model)
print(f'  shape_inference: {len(inferred.graph.value_info)} intermediate shapes inferred')

# Save model
onnx.save(onnx_model, 'pipeline_model.onnx')
import os
model_size = os.path.getsize('pipeline_model.onnx') / 1024
print(f'  Model size: {model_size:.1f} KB')

# STAGE 4: Optimize (try simplifier)
print('\n--- STAGE 4: Optimize ---')
nodes_before = len(onnx_model.graph.node)
try:
    from onnxsim import simplify
    optimized_model, check = simplify(onnx_model)
    if check:
        onnx.save(optimized_model, 'pipeline_model_opt.onnx')
        nodes_after = len(optimized_model.graph.node)
        opt_size = os.path.getsize('pipeline_model_opt.onnx') / 1024
        print(f'  onnx-simplifier: {nodes_before} -> {nodes_after} nodes')
        print(f'  Size: {model_size:.1f} -> {opt_size:.1f} KB')
        deploy_path = 'pipeline_model_opt.onnx'
    else:
        print('  Simplification check failed, using original')
        deploy_path = 'pipeline_model.onnx'
except ImportError:
    try:
        import onnxoptimizer as opt
        optimized_model = opt.optimize(onnx_model)
        onnx.save(optimized_model, 'pipeline_model_opt.onnx')
        nodes_after = len(optimized_model.graph.node)
        print(f'  onnxoptimizer: {nodes_before} -> {nodes_after} nodes')
        deploy_path = 'pipeline_model_opt.onnx'
    except ImportError:
        print('  No optimizer available, using original model')
        deploy_path = 'pipeline_model.onnx'

# STAGE 5: Deploy & Verify
print('\n--- STAGE 5: Deploy (ORT Inference) ---')
sess = ort.InferenceSession(deploy_path, providers=['CPUExecutionProvider'])

X_test_f32 = X_test.astype(np.float32)
onnx_preds = sess.run(None, {'X': X_test_f32})[0].flatten()
onnx_acc = accuracy_score(y_test, onnx_preds)
print(f'  ONNX accuracy: {onnx_acc:.4f}')
print(f'  Matches sklearn: {np.allclose(clf.predict(X_test), onnx_preds)}')

# Benchmark
n_runs = 200
start = time.perf_counter()
for _ in range(n_runs):
    sess.run(None, {'X': X_test_f32})
elapsed = (time.perf_counter() - start) * 1000
print(f'  Throughput: {n_runs * len(X_test_f32) / (elapsed/1000):.0f} samples/sec')
print(f'  Avg latency: {elapsed / n_runs:.2f} ms per batch')

print('\n' + '=' * 60)
print('PIPELINE COMPLETE')
print('=' * 60)

<a id='section-13'></a>
## Section 13: Summary

### What We Practiced

| Exercise | Ecosystem Component | Key API |
|:---|:---|:---|
| Build from scratch | `onnx` library | `helper.make_model()`, `helper.make_node()` |
| Convert sklearn | `skl2onnx` converter | `convert_sklearn()` |
| Convert PyTorch | `torch.onnx` exporter | `torch.onnx.export()` |
| Run inference | ONNX Runtime | `InferenceSession.run()` |
| Inspect graphs | `onnx` API | `model.graph.node`, `numpy_helper` |
| Visualize structure | matplotlib + networkx | Custom `onnx_to_networkx()` |
| Simplify models | onnx-simplifier | `simplify()` |
| Benchmark | ONNX Runtime | `time.perf_counter()` loops |
| End-to-end pipeline | All components | Train → Convert → Validate → Optimize → Deploy |

### Key Takeaways

1. **The ONNX Python API** (`onnx.helper`) enables programmatic model construction — useful for testing, debugging, and understanding the format at the lowest level.

2. **Converters** (`skl2onnx`, `torch.onnx`) bridge the gap between framework-specific models and the universal ONNX format. Always verify numerical equivalence after conversion: $\|f_{\text{fw}}(\mathbf{x}) - f_{\text{onnx}}(\mathbf{x})\|_\infty \leq \epsilon$.

3. **ONNX Runtime** provides a uniform inference API across models from different frameworks. The `InferenceSession` handles graph optimization, memory allocation, and EP dispatch transparently.

4. **Model simplification** removes redundant operations (identity, constant expressions, shape computations), reducing graph complexity by 20-50% on typical exports without affecting accuracy.

5. **Programmatic inspection** of the ONNX graph (nodes, initializers, shapes) is essential for debugging, documentation, and integration — and can be fully automated for CI/CD pipelines.

6. **Performance benchmarking** should include warm-up, multiple runs, and percentile statistics (p50, p95, p99) to capture the full latency distribution.

---

**Previous:** [ONNX Ecosystem Overview — Deep Dive](./ONNX_Ecosystem_Overview_Deep_Dive.ipynb) | **Next:** [Installation and Setup](../04_Installation_and_Setup/)